In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────────
df = pd.read_csv("../DATASET/dataset_expandido.csv", low_memory=False)

# ─────────────────────────────────────────────
# PRÉ-PROCESSAMENTO
# ─────────────────────────────────────────────
have_worked_cols = [
    'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith',
]

def build_binary_matrix(df, cols):
    all_techs = set()
    for col in cols:
        for val in df[col].dropna():
            if val != 'Sem dado':
                all_techs.update([t.strip() for t in str(val).split(';')])
    all_techs = sorted(all_techs)
    matrix = pd.DataFrame(0, index=df.index, columns=all_techs)
    for col in cols:
        for idx, val in df[col].items():
            if pd.notna(val) and val != 'Sem dado':
                for tech in [t.strip() for t in str(val).split(';')]:
                    if tech in matrix.columns:
                        matrix.at[idx, tech] = 1
    return matrix

print('A construir matriz utilizador-tecnologia...')
user_tech_matrix = build_binary_matrix(df, have_worked_cols)
valid_mask       = user_tech_matrix.sum(axis=1) > 0
user_tech_matrix = user_tech_matrix[valid_mask]
df_valid         = df.loc[user_tech_matrix.index].copy()
print(f'Utilizadores válidos : {len(user_tech_matrix)}')
print(f'Tecnologias únicas   : {user_tech_matrix.shape[1]}')

# ─────────────────────────────────────────────
# SIMILARIDADE TECH x TECH
# ─────────────────────────────────────────────
print('A calcular similaridade entre tecnologias...')
tech_similarity_df = pd.DataFrame(
    cosine_similarity(user_tech_matrix.T),
    index=user_tech_matrix.columns,
    columns=user_tech_matrix.columns
)
print(f'Matriz tech x tech: {tech_similarity_df.shape}')

# ─────────────────────────────────────────────
# RECOMENDAÇÃO
# ─────────────────────────────────────────────
def recommend_for_developer(user_idx, user_tech_matrix, tech_similarity_df, top_n=5):
    user_row       = user_tech_matrix.loc[user_idx]
    used_techs     = user_row[user_row == 1].index.tolist()
    not_used_techs = user_row[user_row == 0].index.tolist()
    scores = {}
    for tech in not_used_techs:
        if tech in tech_similarity_df.index:
            scores[tech] = tech_similarity_df.loc[tech, used_techs].mean()
    results = pd.Series(scores).sort_values(ascending=False).head(top_n)
    return used_techs, results

# ─────────────────────────────────────────────
# AVALIAÇÃO — HIT RATE
# ─────────────────────────────────────────────
def avaliar_sistema(user_tech_matrix, tech_similarity_df, n_test=50, top_n=5):
    hits, total = 0, 0
    valid_users = user_tech_matrix.index.tolist()
    test_sample = np.random.choice(len(valid_users), size=n_test, replace=False)
    for i in test_sample:
        user_idx     = valid_users[i]
        user_row     = user_tech_matrix.loc[user_idx]
        langs_usadas = user_row[user_row == 1].index.tolist()
        if len(langs_usadas) < 2:
            continue
        lang_escondida = np.random.choice(langs_usadas)
        user_tech_temp = user_tech_matrix.copy()
        user_tech_temp.at[user_idx, lang_escondida] = 0
        _, recomendacoes = recommend_for_developer(user_idx, user_tech_temp, tech_similarity_df, top_n)
        if lang_escondida in recomendacoes.index:
            hits += 1
        total += 1
    return hits / total if total > 0 else 0

print('A calcular Hit Rate...')
hit_rate = avaliar_sistema(user_tech_matrix, tech_similarity_df, n_test=50)
print(f'Hit Rate: {hit_rate:.2%}')

# ─────────────────────────────────────────────
# ESCOLHER UTILIZADOR SIMPLES DO DATASET
# ─────────────────────────────────────────────
linguagens_simples = {
    'JavaScript', 'Python', 'TypeScript', 'SQL', 'Bash/Shell',
    'Java', 'Go', 'Rust', 'HTML/CSS', 'C#', 'Kotlin', 'PHP', 'Ruby', 'Swift', 'C++'
}
valid_users    = user_tech_matrix.index.tolist()
user_escolhido = None
for uid in valid_users:
    techs = set(user_tech_matrix.loc[uid][user_tech_matrix.loc[uid] == 1].index)
    if techs.issubset(linguagens_simples) and 3 <= len(techs) <= 5:
        user_escolhido = uid
        break
if user_escolhido is None:
    user_escolhido = valid_users[0]

# ─────────────────────────────────────────────
# GERAR DASHBOARD — Plotly
# ─────────────────────────────────────────────
def gerar_dashboard_plotly(user_idx, user_tech_matrix, tech_similarity_df, df_meta, top_n=5):
    used_techs, top_recs_series = recommend_for_developer(
        user_idx, user_tech_matrix, tech_similarity_df, top_n)

    meta = df_meta.loc[user_idx]
    print('=' * 55)
    print('PERFIL DO PROGRAMADOR')
    print('=' * 55)
    print(f'  DevType       : {meta.get("DevType", "N/A")}')
    print(f'  Anos de código: {meta.get("YearsCodePro", "N/A")}')
    print(f'  País          : {meta.get("Country", "N/A")}')
    print(f'  Tecnologias   : {len(used_techs)}')
    print()

    # ── Cores Modernas e Limpas ──
    bg_color    = "#0b1329"
    card_color  = "#1c2541"
    text_light  = "#f8fafc"
    color_curr  = "#4f46e5"  # Índigo para o perfil atual
    color_rec   = "#10b981"  # Verde para as recomendações

    skills_list = sorted(used_techs)
    rec_langs   = list(top_recs_series.index)
    rec_scores  = list(top_recs_series.values)
    
    max_score = max(rec_scores) if rec_scores else 1
    rec_vals_scaled = [s / max_score * 100 for s in rec_scores]

    fig = go.Figure()

    # 1. Traço das competências atuais (Preenche a 100%)
    fig.add_trace(go.Bar(
        x=[100] * len(skills_list),
        y=skills_list,
        name='Your Skills',
        orientation='h',
        marker=dict(color=color_curr),
        text='Active Profile',
        textposition='inside',
        insidetextanchor='start',
        textfont=dict(size=12, color="#e0e7ff"),
        hovertemplate='%{y}<extra></extra>'
    ))

    # 2. Traço de Fundo para as Recomendações (Track escuro)
    fig.add_trace(go.Bar(
        x=[100] * len(rec_langs),
        y=rec_langs,
        orientation='h',
        marker=dict(color=card_color),
        hoverinfo='skip',
        showlegend=False
    ))

    # 3. Traço das Recomendações (Barra de progresso verde)
    fig.add_trace(go.Bar(
        x=rec_vals_scaled,
        y=rec_langs,
        name='Suggested for You',
        orientation='h',
        marker=dict(color=color_rec),
        text=[f'{s:.2f} similarity' for s in rec_scores],
        textposition='inside',
        insidetextanchor='start',
        textfont=dict(size=12, color="#d1fae5"),
        hovertemplate='%{y}: %{x:.1f}% relevância<extra></extra>'
    ))

    # Configuração Simplificada do Layout
    fig.update_layout(
        title=dict(
            text='Language Recommendation Insights',
            x=0.5,
            font=dict(size=20, color=text_light, family='Arial Black'),
        ),
        paper_bgcolor=bg_color,
        plot_bgcolor=bg_color,
        barmode='overlay',  # Mantém o efeito de track nas recomendações
        height=140 + (len(skills_list) + len(rec_langs)) * 40,
        margin=dict(l=130, r=40, t=80, b=40),
        font=dict(color=text_light, family='Arial'),
        
        # Ativa a legenda para o utilizador saber o que é o quê
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=12, color=text_light)
        ),
        
        xaxis=dict(visible=False, range=[0, 105]),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            tickfont=dict(size=13, color=text_light, weight='bold'),
            autorange="reversed"  # Mantém uma ordem limpa de leitura (cima para baixo)
        )
    )

    fig.write_html('../Public/assets/recommendation_system.html')
    fig.show()
    print('Dashboard guardado em ../Public/assets/recommendation_system.html')

# ─────────────────────────────────────────────
# CORRER
# ─────────────────────────────────────────────
gerar_dashboard_plotly(user_escolhido, user_tech_matrix, tech_similarity_df, df_valid)

A construir matriz utilizador-tecnologia...
Utilizadores válidos : 151751
Tecnologias únicas   : 219
A calcular similaridade entre tecnologias...
Matriz tech x tech: (219, 219)
A calcular Hit Rate...
Hit Rate: 30.61%
PERFIL DO PROGRAMADOR
  DevType       : Student
  Anos de código: Sem dado
  País          : Philippines
  Tecnologias   : 3



Dashboard guardado em ../Public/assets/recommendation_system.html
